In [ ]:
# jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from isaacsim.examples.interactive.base_sample import BaseSample
from isaacsim.core.api import World
from isaacsim.robot.manipulators.examples.franka import Franka
from isaacsim.core.utils.nucleus import get_assets_root_path
from isaacsim.core.api.objects import DynamicCuboid
from isaacsim.core.api.robots import Robot

from isaacsim.robot.manipulators.examples.franka.controllers import PickPlaceController
from isaacsim.robot.manipulators.examples.franka.tasks import PickPlace
from isaacsim.robot.wheeled_robots.controllers.wheel_base_pose_controller import WheelBasePoseController
from isaacsim.robot.wheeled_robots.controllers.differential_controller import DifferentialController

from isaacsim.core.api.tasks import BaseTask

import isaacsim.core.utils.stage as stage_utils

import numpy as np

In [ ]:
class HelloWorld(BaseSample):
    def __init__(self) -> None:
        super().__init__()
        # define assets root path
        self._isaac_assets_path = get_assets_root_path()
        
        # define url of assets
        self.jetbot_url = self._isaac_assets_path + "/Isaac/Robots/NVIDIA/Jetbot/jetbot.usd"
        return

    def setup_scene(self):
        world = self.get_world()
        world.add_task(PickPlace(name="awesome_task"))
        return
    
    # right after the scene setup 
    async def setup_post_load(self):
        self._world = self.get_world()

        task_params = self._world.get_task("awesome_task").get_params()
        self._franka = self._world.scene.get_object(task_params["robot_name"]["value"])
        self._cube_name = task_params["cube_name"]["value"]
        
        # initializing a pick and place controller
        self._frankactrl = PickPlaceController(
            name ="pick_place_controller",
            gripper=self._franka.gripper,
            robot_articulation=self._franka,
        )
        self._world.add_physics_callback("franka_step", callback_fn = self.franka_step)
        
        await self._world.play_async()
        return

    async def setup_post_reset():
        #franka
        self._frankactrl.reset()
        self._franka.gripper.set_joint_positions(self._franka.gripper.joint_opened_positions)
        
        await self._world.play_async()
        return

    def franka_step(self, step_size):
        current_observations = self._world.get_observations()
        actions = self._frankactrl.forward(
            picking_position=current_observations[self._cube_name]["position"],
            placing_position=current_observations[self._cube_name]["target_position"],
            current_joint_positions=current_observations[self._franka.name]["joint_positions"],
        )
        self._franka.apply_action(actions)
        if self._frankactrl.is_done():
            self._world.pause()
        return


In [ ]:
await HelloWorld().load_world_async()